In [4]:
import matplotlib.pyplot as plt
import random
from sklearn.datasets import make_classification
import numpy as np
import pandas as pd
import seaborn as sns


from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LassoCV, LogisticRegression
from sklearn.metrics import log_loss
from sklearn.ensemble import RandomForestClassifier

import sys

sys.path.append("./..")

from src.generate_data import generate_dataset

rng = random.Random(213)

In [7]:
X, y, metadata = generate_dataset()

[6, 2, 1, 3, 8]

informative [0, 1, 2, 3, 4, 5]
redundant [6, 7]
correlated [8]
noise [9, 10, 11]
pure_noise [12, 13, 14, 15, 16, 17, 18, 19]


In [70]:
seed = rng.randint(1, 10000)
X, y, feature_types = generate_dataset(n_samples=500, n_features=20, random_state=seed)
num_of_informative_features = len(feature_types["informative"])

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=seed
)

# Store results
selected_features = {
    "everything": list(range(20)),
    "informative": metadata["informative"],
}

# --- 1. LASSO ---
lasso = LassoCV(cv=5, random_state=seed, max_iter=10000).fit(X_train, y_train)
lasso_selected = np.where(lasso.coef_ != 0)[0].tolist()
selected_features["lasso"] = lasso_selected

# --- 2. Mutual Information ---
mi_scores = mutual_info_classif(X_train, y_train, random_state=seed)
mi_selected = np.argsort(mi_scores)[-num_of_informative_features:].tolist()
selected_features["mutual_info"] = mi_selected

# --- 3. Correlation ---
correlations = []
for i in range(X_train.shape[1]):
    corr = np.corrcoef(X_train[:, i], y_train)[0, 1]
    correlations.append((i, abs(corr)))

correlated = [
    i
    for i, _ in sorted(correlations, key=lambda x: -x[1])[:num_of_informative_features]
]
selected_features["correlation"] = correlated


# --- 4. BIC Wrapper (Logistic Regression with feature subsets) ---
def bic_score(model, X, y):
    """Calculate BIC for a fitted logistic regression model"""
    n = len(y)
    k = X.shape[1]
    prob = model.predict_proba(X)[:, 1]
    ll = -log_loss(y, prob, normalize=False)  # log-likelihood
    bic = k * np.log(n) - 2 * ll
    return bic


def bic_forward_selection(X, y, max_features=None):
    selected = []
    remaining = list(range(X.shape[1]))
    best_bic = np.inf

    while remaining:
        scores = []
        for feat in remaining:
            candidate = selected + [feat]
            model = LogisticRegression(solver="liblinear", max_iter=10000).fit(
                X[:, candidate], y
            )
            bic = bic_score(model, X[:, candidate], y)
            scores.append((bic, feat))

        # Find the best new feature to add
        scores.sort()
        best_new_bic, best_feat = scores[0]

        # Stop if BIC does not improve
        if best_new_bic < best_bic:
            best_bic = best_new_bic
            selected.append(best_feat)
            remaining.remove(best_feat)
            if max_features and len(selected) >= max_features:
                break
        else:
            break

    return selected


bic_selected = bic_forward_selection(X_train, y_train)
selected_features["bic"] = bic_selected

# --- 5. Random Forest Feature Importance ---
rf = RandomForestClassifier(n_estimators=100, random_state=seed)
rf.fit(X_train, y_train)

importances = rf.feature_importances_
rf_selected = np.argsort(importances)[-num_of_informative_features:].tolist()
selected_features["random_forest"] = rf_selected

# --- Print results ---
print()
for method, features in selected_features.items():
    print(f"{method}: selected feature indices: {features}")

[3, 6, 1, 8, 2]

informative [0, 1, 2]
redundant [3, 4, 5, 6, 7, 8]
correlated [9]
noise [10, 11, 12, 13, 14, 15, 16, 17]
pure_noise [18, 19]

everything: selected feature indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
informative: selected feature indices: [0, 1, 2, 3, 4, 5]
lasso: selected feature indices: [2, 4, 5, 19]
mutual_info: selected feature indices: [16, 6, 8]
correlation: selected feature indices: [4, 5, 0]
bic: selected feature indices: [4, 7, 2]
random_forest: selected feature indices: [6, 16, 5]


In [71]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Split your data (assumes X and y are already defined)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Create and train an SVM classifier
svm_clf = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)
svm_clf.fit(X_train, y_train)

# 3. Create and train a Random Forest classifier
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train, y_train)

# 4. Predict and evaluate
svm_preds = svm_clf.predict(X_test)
rf_preds = rf_clf.predict(X_test)

print("SVM Accuracy:", accuracy_score(y_test, svm_preds))
print("Random Forest Accuracy:", accuracy_score(y_test, rf_preds))

SVM Accuracy: 0.89
Random Forest Accuracy: 0.91


In [72]:
def select_features(X, columns, key):
    if key not in columns:
        raise Exception("key not in dictionary")
    return X[:, columns[key]]


select_features(X, selected_features, "bic")

array([[-1.38391942,  0.87318655,  2.05868932],
       [-1.97894724, -0.02456119,  0.97887764],
       [-2.36357399, -0.53115499,  0.4066328 ],
       ...,
       [ 0.92492008, -0.09983365,  0.3291578 ],
       [ 1.59146106,  1.33802411, -0.14626843],
       [-0.63746085, -0.86219089,  2.52841053]])

In [73]:
selected_features

{'everything': [0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19],
 'informative': [0, 1, 2, 3, 4, 5],
 'lasso': [2, 4, 5, 19],
 'mutual_info': [16, 6, 8],
 'correlation': [4, 5, 0],
 'bic': [4, 7, 2],
 'random_forest': [6, 16, 5]}

In [75]:
for key, value in selected_features.items():
    X_prim = select_features(X, selected_features, key)

    X_train, X_test, y_train, y_test = train_test_split(
        X_prim, y, test_size=0.2, random_state=42
    )

    svm_clf = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)
    svm_clf.fit(X_train, y_train)

    rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_clf.fit(X_train, y_train)

    svm_preds = svm_clf.predict(X_test)
    rf_preds = rf_clf.predict(X_test)

    print(f"{key} SVM Accuracy:", accuracy_score(y_test, svm_preds))
    print(f"{key} Random Forest Accuracy:", accuracy_score(y_test, rf_preds))

everything SVM Accuracy: 0.89
everything Random Forest Accuracy: 0.91
informative SVM Accuracy: 0.88
informative Random Forest Accuracy: 0.88
lasso SVM Accuracy: 0.89
lasso Random Forest Accuracy: 0.89
mutual_info SVM Accuracy: 0.75
mutual_info Random Forest Accuracy: 0.82
correlation SVM Accuracy: 0.88
correlation Random Forest Accuracy: 0.83
bic SVM Accuracy: 0.89
bic Random Forest Accuracy: 0.92
random_forest SVM Accuracy: 0.86
random_forest Random Forest Accuracy: 0.84
